# Hunde- und Katzenbilder laden und labeln

## Drive in Colab mounten

In [ ]:
use_colab = True

if (use_colab):
  # Google Drive mounten
  from google.colab import drive
  drive.mount("/content/drive")
  base_path = "/content/drive/MyDrive"
else:
  base_path = '..'

## Bilder einlesen

Quelle der Bilddateien:
https://downloads.codingcoursestv.eu/037%20-%20neuronale%20netze/PetImages.zip

### `image_dataset_from_directory()`

Mit `image_dataset_from_directory()` werden Bilder nicht vollständig in den Arbeitsspeicher geladen, sondern während des Trainings batchweise von der Festplatte eingelesen. Die Klassenlabels werden dabei automatisch aus den Ordnernamen (z. B. `Cat` und `Dog`) erzeugt. Dadurch können auch sehr große Bilddatensätze effizient mit `model.fit()` verarbeitet werden.

In [ ]:
import os
import tensorflow as tf

dataset_path = os.path.join(base_path, "Datensätze", "PetImages")

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32,
    label_mode="binary"
)

print(train_ds.class_names)

## Transfer Learning mit VGG16


### Die Bilder werden für das VGG16 vorverarbeitet

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Wir schicken unsere Daten nur einmal durch das VGG16-Modell. Die Ausgabe des VGG16 nehmen wir dann als Eingabe für das Dense-Netz. So können wir das Training beschleunigen.



In [ ]:
from tqdm.notebook import tqdm

vgg16_model = VGG16(include_top=False, input_shape=(224, 224, 3))
vgg16_model.trainable = False

# Trainingsfeatures erzeugen
features = []
labels = []

for images, y in tqdm(train_ds, total=len(train_ds)):
    f = vgg16_model(images, training=False)

    features.append(f)
    labels.append(y)

X_nach_vgg = tf.concat(features, axis=0)
y = tf.concat(labels, axis=0)

In [ ]:
# Validierungsfeatures erzeugen
val_features = []
val_labels = []

for images, y in tqdm(train_ds, total=len(train_ds)):
    f = vgg16_model(images, training=False)

    features.append(f)
    labels.append(y)


X_val_vgg = tf.concat(val_features, axis=0)
y_val = tf.concat(val_labels, axis=0)

Jetzt brauchen wir kein VGG16 mehr in unserem Model.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(Flatten())
model.add(Dense(1024, activation="relu"))
model.add(Dense(16, activation="relu"))
model.add(Dense(1, activation="sigmoid"))
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])


Model trainieren

In [ ]:
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import EarlyStopping

# Daten mischen
X_nach_vgg, y = shuffle(X_nach_vgg, y, random_state=42)

# Early Stopping
early_stopping = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_vgg,
    y_train,
    epochs=50,
    batch_size=300,
    validation_data=(X_val_vgg, y_val),
    callbacks=[early_stopping]
)

In [ ]:
file_path = os.path.join(base_path, "ErkennungHundKatze.keras")
model.save(file_path)